# UTS Deep Learning - Reproduksi Gabungan

Jalankan eksperimen klasifikasi, RUL, dan klaster dari notebook mandiri. Panduan folder dan dependensi tersedia di README.md.

## Sel 1 · Lokasi data dan checkout

Notebook ini menjalankan ketiga notebook tugas dari checkout repositori. Saat memakai Colab, siapkan checkout di runtime dengan git clone, lalu gunakan UTS_REPO_DIR bila lokasi belum terdeteksi. Dataset hanya dibaca dari Drive atau direktori UTS_DATA_DIR.

In [ ]:
import os
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None

cwd = Path.cwd()
calon = [cwd, *cwd.parents]
if drive is not None:
    calon.extend(p for p in Path("/content").iterdir() if p.is_dir())
path_repo = os.environ.get("UTS_REPO_DIR")
if path_repo:
    REPO = Path(path_repo).resolve()
else:
    REPO = next((p for p in calon if (p/"kode/tugas3/Klaster_Material.ipynb").is_file()), None)
if REPO is None or not (REPO/"kode/tugas1/Klasifikasi_Cacat.ipynb").is_file():
    raise FileNotFoundError("Siapkan checkout repositori, lalu set UTS_REPO_DIR ke folder tersebut.")

if drive is not None:
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
    simpan_data = Path("/content/drive/MyDrive/UTS_DL/data_uts_dl")
    simpan_hasil = Path("/content/hasil_reproduksi_uts")
else:
    simpan_data = REPO / "data_uts_dl"
    simpan_hasil = REPO / "hasil_reproduksi"

DATA = Path(os.environ.get("UTS_DATA_DIR", simpan_data)).resolve()
HASIL = Path(os.environ.get("UTS_HASIL_DIR", simpan_hasil)).resolve()
if not DATA.is_dir():
    raise FileNotFoundError(f"Periksa lokasi dataset: {DATA}")
HASIL.mkdir(parents=True, exist_ok=True)
print("Repo:", REPO, "| Data:", DATA, "| Keluaran:", HASIL)

## Sel 2 · Fungsi reproduksi dan penggabungan

Setiap notebook dijalankan dalam konteks tersendiri dan memverifikasi prediksinya. Notebook ini menyusun sampul dari hasil eksekusi, menggabungkan PDF dalam urutan 1, 2, 3, lalu memberikan nomor halaman berkelanjutan.

In [ ]:
%pip -q install reportlab pypdf

import gc
import io
import json
import shutil
from xml.sax.saxutils import escape

import pandas as pd

from pypdf import PdfReader, PdfWriter
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.pdfgen import canvas
from reportlab.platypus import Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle

NOTEBOOK = [
    "kode/tugas1/Klasifikasi_Cacat.ipynb",
    "kode/tugas2/Prediksi_RUL.ipynb",
    "kode/tugas3/Klaster_Material.ipynb",
]
BAGIAN = [
    "01_klasifikasi_cacat.pdf", "02_prediksi_rul.pdf", "03_klaster_material.pdf",
]
PREDIKSI = ["prediksi_klasifikasi.csv", "prediksi_rul.csv", "cluster.csv"]


def jalankan_notebook(nomor, tujuan):
    sumber = REPO / NOTEBOOK[nomor - 1]
    nb = json.loads(sumber.read_text(encoding="utf-8"))
    env = {"DATA": DATA, "HASIL": tujuan, "REPO": REPO}
    for sel in nb["cells"]:
        if sel["cell_type"] == "code" and sel["id"] in {f"kode-{i}" for i in range(1, 6)}:
            kode = "".join(sel["source"])
            kode = "\n".join(l for l in kode.splitlines() if not l.startswith("%pip "))
            exec(compile(kode, f"{sumber.name}:{sel['id']}", "exec"), env)
    return env[f"hasil_tugas{nomor}"]


def gabungkan_laporan(hasil):
    folder_bagian = hasil / "laporan_bagian"
    stats1 = json.loads((hasil/"artifacts/tugas1/hasil.json").read_text(encoding="utf-8"))
    stats2 = json.loads((hasil/"artifacts/tugas2/hasil.json").read_text(encoding="utf-8"))
    stats3 = json.loads((hasil/"artifacts/tugas3/verifikasi_tugas3.json").read_text(encoding="utf-8"))
    f1 = stats1["ensemble"]["f1"]
    rul = stats2["model"][stats2["final"]]["evaluasi"]
    k = stats3["klaster"]
    klaster = pd.read_csv(hasil/"cluster.csv")
    pembanding = pd.read_csv(hasil/"artifacts/tugas3/t3_pembanding.csv")
    assert klaster.id_sampel.equals(pembanding.id_sampel)
    model3 = "DEC dengan regularisasi rekonstruksi" if klaster.cluster.equals(
        pembanding.DEC
    ) else "K-means pada embedding gabungan"
    def jumlah(x):
        return f"{x:,}".replace(",", ".")

    def angka(x, digit):
        return f"{x:.{digit}f}".replace(".", ",")

    biru = colors.HexColor("#173452")
    teal = colors.HexColor("#166254")
    gaya = getSampleStyleSheet()
    gaya.add(ParagraphStyle("awal", fontName="Helvetica-Bold", fontSize=10, leading=13,
                             textColor=teal, spaceAfter=9))
    gaya.add(ParagraphStyle("judul_laporan", fontName="Helvetica-Bold", fontSize=25, leading=29,
                             textColor=biru, spaceAfter=13))
    gaya.add(ParagraphStyle("isi_laporan", fontName="Helvetica", fontSize=10, leading=14.5,
                             textColor=biru, spaceAfter=9))
    gaya.add(ParagraphStyle("tabel_laporan", parent=gaya["isi_laporan"], fontSize=9, leading=12,
                             spaceAfter=0))
    gaya.add(ParagraphStyle("kepala_laporan", parent=gaya["tabel_laporan"], fontName="Helvetica-Bold",
                             textColor=colors.white))
    gaya.add(ParagraphStyle("sub_laporan", fontName="Helvetica-Bold", fontSize=12, leading=15,
                             textColor=biru, spaceBefore=8, spaceAfter=5))

    def p(teks, jenis="isi_laporan"):
        return Paragraph(escape(str(teks)), gaya[jenis])

    rows = [
        ["Tugas", "Model utama", "Hasil evaluasi"],
        ["1. Klasifikasi", "Ensemble fusi logit dengan smoothing dan augmentasi"
         if stats1["final"] == "logit_augmentasi" else f"Ensemble {stats1['final']}",
         f"Macro-F1 {angka(f1, 3)} pada {jumlah(stats1['split']['evaluasi'])} unit evaluasi"],
        ["2. RUL", f"Ensemble {stats2['final'].upper()}",
         f"RMSE {angka(rul['rmse'], 2)} siklus pada {rul['n']} mesin evaluasi"],
        ["3. Klaster", model3,
         f"{k} klaster dari {jumlah(stats3['sampel'])} sampel"],
    ]
    tabel = Table([[p(x, "kepala_laporan" if i == 0 else "tabel_laporan") for x in row]
                  for i, row in enumerate(rows)], colWidths=[108, 195, 197], hAlign="LEFT")
    tabel.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), biru),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.HexColor("#EDF4F7"), colors.white]),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("TOPPADDING", (0, 0), (-1, -1), 8),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 8),
    ]))

    alur = [
        p("UTS DEEP LEARNING  /  LAPORAN KELOMPOK", "awal"),
        p("Klasifikasi cacat, prediksi RUL, dan klaster material", "judul_laporan"),
        p("Tiga eksperimen memakai dataset UTS yang disediakan. Jaringan neural dilatih dari awal "
          "tanpa data eksternal atau bobot pralatih."),
        tabel, Spacer(1, 10),
        p("Rancangan evaluasi", "sub_laporan"),
        p("Klasifikasi memakai evaluasi berlabel inspektur yang terpisah dari latih dan pengembangan. "
          "Regresi membagi data per mesin, dengan kelompok kalibrasi tersendiri untuk interval. "
          "Klaster dievaluasi menggunakan ukuran internal dan kestabilan partisi. Skor uji tersembunyi "
          "ditentukan pada penilaian pengajar."),
        p("Reproduksi", "sub_laporan"),
        p("Jalankan jalankan_semua() pada notebook kode/UTS_Deep_Learning.ipynb. Fungsi menjalankan "
          "tiga notebook tugas, memverifikasi CSV, dan menyusun PDF ini. Referensi CPU memakai seed "
          "ensemble 17, 29, dan 43. Pencarian dan visualisasi klaster menggunakan seed yang dicatat dalam notebook."),
        p(f"Versi referensi: Python {stats1['versi']['Python']}, PyTorch {stats1['versi']['torch']}, "
          f"NumPy {stats1['versi']['numpy']}, pandas {stats1['versi']['pandas']}, "
          f"scikit-learn {stats1['versi']['scikit-learn']}. Cara pemasangan dan jalur data tercantum di kode/README.md."),
        p("Sumber", "sub_laporan"),
        p("Soal UTS Deep Learning dan kamus_data.md. Referensi interval: Angelopoulos dan Bates, "
          "A Gentle Introduction to Conformal Prediction and Distribution-Free Uncertainty Quantification, "
          "arXiv:2107.07511. Referensi klaster dan t-SNE dicantumkan pada Tugas 3."),
    ]

    def kaki_sampul(kanvas, doc):
        kanvas.setFont("Helvetica", 8)
        kanvas.setFillColor(colors.HexColor("#546476"))
        kanvas.setStrokeColor(colors.HexColor("#D3E1E8"))
        kanvas.line(45, 40, 550, 40)
        kanvas.drawString(45, 27, "UTS DEEP LEARNING  /  EKSPERIMEN MULTIMODAL")
        kanvas.drawRightString(550, 27, "1")

    cover = io.BytesIO()
    SimpleDocTemplate(cover, pagesize=A4, leftMargin=45, rightMargin=45, topMargin=41,
                      bottomMargin=55, title="UTS Deep Learning").build(
                          alur, onFirstPage=kaki_sampul, onLaterPages=kaki_sampul,
                      )
    cover.seek(0)
    writer = PdfWriter()
    writer.append(cover)
    nomor = len(writer.pages) + 1
    for nama, file in zip(("Tugas 1 - Klasifikasi", "Tugas 2 - RUL", "Tugas 3 - Klaster"), BAGIAN):
        awal = len(writer.pages)
        for page in PdfReader(folder_bagian/file).pages:
            lebar, tinggi = float(page.mediabox.width), float(page.mediabox.height)
            overlay = io.BytesIO()
            layer = canvas.Canvas(overlay, pagesize=(lebar, tinggi))
            layer.setFillColor(colors.white)
            layer.rect(lebar-100, 20, 75, 16, fill=1, stroke=0)
            layer.setFillColor(colors.HexColor("#546476"))
            layer.setFont("Helvetica", 8)
            layer.drawRightString(lebar-45, 27, str(nomor))
            layer.save()
            overlay.seek(0)
            page.merge_page(PdfReader(overlay).pages[0])
            writer.add_page(page)
            nomor += 1
        writer.add_outline_item(nama, page_number=awal)
    assert len(writer.pages) <= 18
    writer.add_metadata({"/Title": "UTS Deep Learning - Laporan Kelompok"})
    path = hasil / "laporan.pdf"
    writer.write(path)
    return path


def jalankan_semua():
    for nomor in (1, 2):
        jalankan_notebook(nomor, HASIL)
        gc.collect()
    folder = HASIL / "artifacts" / "tugas3"
    jalankan_notebook(3, folder)
    shutil.copyfile(folder / "cluster.csv", HASIL / "cluster.csv")
    shutil.copyfile(folder / BAGIAN[2], HASIL / "laporan_bagian" / BAGIAN[2])
    hasil_pdf = gabungkan_laporan(HASIL)
    print("Verifikasi gabungan LULUS")
    for nama in PREDIKSI:
        print(f"{nama}: {len(pd.read_csv(HASIL / nama)):,} baris")
    print(f"{hasil_pdf} | {len(PdfReader(hasil_pdf).pages)} halaman")
    return hasil_pdf

## Sel 3 · Eksekusi ketiga tugas

Pemanggilan jalankan_semua() mereproduksi ketiga CSV serta laporan. Checkpoint CPU Tugas 1 dan 2 dipakai bila cocok. Tugas 3 dilatih ulang. Hasil referensi menggunakan CPU.

In [ ]:
laporan_gabungan = jalankan_semua()

## Sel 4 · Unduh hasil gabungan

ZIP memuat laporan.pdf, ketiga CSV, dan PDF bagian. Tambahkan folder kode serta kontribusi.md yang telah diisi dari repositori kelompok.

In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

paket = HASIL / "hasil_pengumpulan.zip"
with ZipFile(paket, "w", compression=ZIP_DEFLATED) as z:
    for nama in [*PREDIKSI, "laporan.pdf"]:
        z.write(HASIL / nama, nama)
    for nama in BAGIAN:
        z.write(HASIL / "laporan_bagian" / nama, f"laporan_bagian/{nama}")

print("Arsip:", paket)
if drive is not None:
    from google.colab import files
    files.download(str(paket))